# Notebook 11 — Optimization and Gradient Descent

**Module:** 02 — Mathematics for Biology  
**Notebook:** 11 of 12  
**Prerequisites:** NB02, NB03  
**Time estimate:** 90 minutes

---
## Step 1 — Motivation

Every time you fit a model to data — a logistic growth curve (NB04), a Michaelis-Menten
curve (Module 01 NB11), a neural network weight — you are solving an optimization
problem. Gradient descent is the algorithm underlying nearly all parameter estimation
in modern computational biology. Understanding it from scratch is essential before
Module 13 (Machine Learning) and Module 14 (Deep Learning).

---
## Step 2 — Intuition

Imagine you are blindfolded in a hilly landscape, trying to find the lowest valley.
Gradient descent: at each step, feel which direction is downhill, take a step in
that direction, repeat.

The *gradient* points uphill; subtract it to go downhill. The *learning rate*
(step size $\eta$) controls how far you step. Too large: overshoot and diverge.
Too small: take forever to converge.

---
## Step 3 — Biological Background

**Where optimization appears in biology:**

| Task | Loss function | Method |
|------|-------------|--------|
| Curve fitting (e.g. Michaelis-Menten) | Sum of squared residuals | `scipy.curve_fit` (Levenberg-Marquardt, a GD variant) |
| Maximum likelihood estimation | Negative log-likelihood | Gradient-based (L-BFGS, Adam) |
| Neural network training (Module 14) | Cross-entropy / MSE | Stochastic gradient descent, Adam |
| Sequence alignment scoring | Gap-penalised score | Dynamic programming (NB08 in Module 08) |

**Evolutionary analogy:** natural selection is a form of optimization —
it minimises reproductive failure on the fitness landscape. But unlike gradient
descent, it is stochastic, population-based, and can escape local optima.

---
## Step 4 — Mathematical Explanation

**Gradient descent update rule:**
$$\theta_{t+1} = \theta_t - \eta \nabla_{\theta} L(\theta_t)$$

where $L(\theta)$ is the loss function and $\eta > 0$ is the learning rate.

**Gradient (multivariate):** $\nabla L = [\partial L/\partial \theta_1, \ldots, \partial L/\partial \theta_n]^T$

**Convergence for convex functions:** with appropriate $\eta$, GD converges to the
global minimum. For non-convex functions (neural networks), it may converge to a
local minimum.

**Stochastic GD (SGD):** approximate $\nabla L$ using a mini-batch of $m$ samples.
Noisier but faster and can escape shallow local minima.

**Momentum (Adam, RMSProp):** accumulate a running average of past gradients to
smooth oscillations and accelerate convergence.

---
## Step 5 — Computational Explanation

**Numerical gradient (from NB02):**
```python
grad[i] = (L(theta + eps_i) - L(theta - eps_i)) / (2 * eps)
```
For 2D problems this is practical. For high-dimensional neural networks, automatic
differentiation (Module 14) is required.

**Convergence detection:**
- Track loss $L(\theta_t)$ over iterations
- Stop when $|L_{t+1} - L_t| < \epsilon_{\text{tol}}$ (or after max iterations)
- Gradient norm $\|\nabla L\|$ → 0 at a stationary point

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
# Cell 6.1 — Gradient descent from scratch: 1D example
def gradient_descent_1d(loss_fn, grad_fn, theta0: float, lr: float = 0.1,
                         n_iter: int = 200, tol: float = 1e-8):
    """
    1D gradient descent minimising loss_fn starting from theta0.

    Returns
    -------
    theta_hist : list of parameter values
    loss_hist  : list of loss values
    """
    theta = theta0
    theta_hist, loss_hist = [theta], [loss_fn(theta)]
    for _ in range(n_iter):
        g = grad_fn(theta)
        theta_new = theta - lr * g
        loss_hist.append(loss_fn(theta_new))
        theta_hist.append(theta_new)
        if abs(theta_new - theta) < tol:
            break
        theta = theta_new
    return theta_hist, loss_hist

# Quadratic loss: L(θ) = (θ - 3)^2
loss_quad = lambda theta: (theta - 3)**2
grad_quad = lambda theta: 2 * (theta - 3)

theta_h, loss_h = gradient_descent_1d(loss_quad, grad_quad, theta0=-2.0, lr=0.2)
print(f"Converged at θ = {theta_h[-1]:.6f} (true minimum: 3.0)")
print(f"Iterations: {len(theta_h)}")

In [ ]:
# Cell 6.2 — Fitting Michaelis-Menten by gradient descent (2D)
# MM model: v(S) = Vmax * S / (Km + S)
# Loss: mean squared error

rng = np.random.default_rng(42)
Vmax_true, Km_true = 10.0, 2.0
S_obs = np.array([0.1, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0])
v_obs = (Vmax_true * S_obs / (Km_true + S_obs)) + rng.normal(0, 0.3, len(S_obs))

def mm_model(S, Vmax, Km):
    return Vmax * S / (Km + S)

def mse_loss(params):
    Vmax, Km = params
    v_pred = mm_model(S_obs, Vmax, Km)
    return np.mean((v_pred - v_obs)**2)

def numerical_gradient(loss_fn, params, eps=1e-6):
    params = np.asarray(params, dtype=float)
    grad = np.zeros_like(params)
    for i in range(len(params)):
        p_plus, p_minus = params.copy(), params.copy()
        p_plus[i] += eps; p_minus[i] -= eps
        grad[i] = (loss_fn(p_plus) - loss_fn(p_minus)) / (2 * eps)
    return grad

# Gradient descent
params = np.array([5.0, 5.0])  # initial guess
lr = 0.05
loss_history = [mse_loss(params)]

for iteration in range(500):
    g = numerical_gradient(mse_loss, params)
    params_new = params - lr * g
    params_new = np.clip(params_new, 0.01, 100)  # enforce positive
    loss_history.append(mse_loss(params_new))
    if np.linalg.norm(params_new - params) < 1e-7:
        break
    params = params_new

Vmax_fit, Km_fit = params
print(f"True:   Vmax={Vmax_true}, Km={Km_true}")
print(f"Fitted: Vmax={Vmax_fit:.4f}, Km={Km_fit:.4f}")
print(f"Final MSE: {loss_history[-1]:.6f}")

In [ ]:
# Cell 6.3 — Learning rate sensitivity
lr_vals = [0.01, 0.1, 0.5, 1.2]
histories = {}
for lr_test in lr_vals:
    _, lh = gradient_descent_1d(loss_quad, grad_quad, theta0=-2.0, lr=lr_test, n_iter=50)
    histories[lr_test] = lh
    print(f"lr={lr_test}:  converged={lh[-1] < 1e-6}  final_loss={lh[-1]:.4e}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Three panels: loss surface, convergence, MM fit
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: 1D loss landscape with GD path
ax = axes[0]
theta_range = np.linspace(-3, 6, 200)
ax.plot(theta_range, loss_quad(theta_range), color="black", lw=2)
ax.scatter(theta_h, loss_h, c=range(len(theta_h)), cmap="plasma",
           s=30, zorder=5, label="GD steps")
ax.set_xlabel("θ"); ax.set_ylabel("L(θ) = (θ-3)²")
ax.set_title("Gradient descent on quadratic loss")
ax.legend()

# Panel 2: loss over iterations for different learning rates
ax = axes[1]
colors_lr = ["steelblue", "green", "orange", "tomato"]
for (lr_test, lh), col in zip(histories.items(), colors_lr):
    ax.semilogy(lh, color=col, lw=1.5, label=f"lr={lr_test}")
ax.set_xlabel("Iteration"); ax.set_ylabel("Loss (log)")
ax.set_title("Learning rate sensitivity")
ax.legend(fontsize=8)

# Panel 3: MM fit result
ax = axes[2]
S_dense = np.linspace(0, 20, 200)
ax.scatter(S_obs, v_obs, color="black", s=40, zorder=5, label="Observed")
ax.plot(S_dense, mm_model(S_dense, Vmax_true, Km_true),
        color="gray", ls="--", lw=1.5, label="True")
ax.plot(S_dense, mm_model(S_dense, Vmax_fit, Km_fit),
        color="steelblue", lw=2, label=f"GD fit (Vmax={Vmax_fit:.2f}, Km={Km_fit:.2f})")
ax.set_xlabel("[S]"); ax.set_ylabel("v")
ax.set_title("Michaelis-Menten: GD fit")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `gradient_descent_2d(loss_fn, theta0, lr, n_iter)` using `numerical_gradient`
   and apply it to minimise $L(\theta_1, \theta_2) = (\theta_1 - 2)^2 + (\theta_2 + 1)^2$.
   Verify the minimum is at $(2, -1)$.
2. For the Michaelis-Menten fit: what is the condition number of the loss landscape
   around the optimum? (Hint: compute the Hessian numerically using second-order
   finite differences, then find the ratio of its largest to smallest eigenvalue.)
3. Compare `gradient_descent` (manual) to `scipy.optimize.minimize(method='L-BFGS-B')`
   on the Michaelis-Menten problem. Which converges faster and to a better solution?
4. Implement *momentum gradient descent*:
   $v_{t+1} = \mu v_t + \eta \nabla L$, $\theta_{t+1} = \theta_t - v_{t+1}$
   with $\mu = 0.9$. Does it converge faster than plain GD on the quadratic?

---
## Quiz — Active Recall

1. Write the gradient descent update rule from memory.
2. What happens if the learning rate is too large? Too small?
3. What is the gradient of $L(\theta) = (\theta - a)^2$? Derive it.
4. What is stochastic gradient descent? Why is it used in machine learning?
5. Name three places in computational biology where gradient-based optimization is used.

---
## Reflection

**Date completed:** ____________________

> *[Can you implement gradient descent from scratch without looking? What is the one thing about learning rate selection that you'd quote in an interview?]*

---
*Next: `12_mini_project_assessment.ipynb`*